In [1]:
from pathlib import Path
import cdsapi 
import yaml

PROJECT_ROOT= Path.cwd().parent

cds_key = Path.home()/".cdsapirc"
if not cds_key.exists():
    raise FileNotFoundError(f"CDS API key not found. Please create a .cdsapirc file in your home directory.")
with open(PROJECT_ROOT/"config.yaml") as f:
    config = yaml.safe_load(f)

turin=config["zones"]["nord"]["cities"][0]
print(f"Downloading ERA5 data for {turin}...")

In [2]:
client=cdsapi.Client()
test_output=PROJECT_ROOT/"data"/"raw"/"weather"/"era5_test_torino.nc"
client.retrieve(
    "reanalysis-era5-single-levels-timeseries",
    {
        "variable": [
            "2m_temperature",
            "surface_solar_radiation_downwards",
            "total_sky_direct_solar_radiation_at_surface",
            "surface_pressure",
            "100m_u_component_of_wind",
            "100m_v_component_of_wind",
        ],
        "location": {"latitude": turin["lat"], "longitude": turin["lon"]},
        "date": ["1991-01-01/2023-12-31"],
        "data_format": "netcdf",
        "download_format": "unarchived",
    },
    str(test_output),
)

print("Saved to:", test_output)
print("Is it actually a zip?", open(test_output, "rb").read(4) == b"PK\x03\x04")


2026-08-22 16:50:23,129 INFO Request ID is 1c952b86-a338-4c9d-a93b-b6b1356f1408


2026-08-22 16:50:23,545 INFO status has been updated to accepted


2026-08-22 16:50:37,285 WARNING Download format not supported for this dataset. Defaulting to zip.


2026-08-22 16:50:44,966 INFO status has been updated to successful


c12446a6eeb574da0a026a1dca675084.zip:   0%|          | 0.00/8.87M [00:00<?, ?B/s]

Saved to: /Users/sristykala/ppa_selection_italy/data/raw/weather/era5_test_torino.nc
Is it actually a zip? True


In [3]:
import zipfile

extract_dir = PROJECT_ROOT / "data" / "raw" / "weather" / "era5_test_torino_extracted"
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(test_output) as zf:
    print(zf.namelist())
    zf.extractall(extract_dir)

nc_file = next(extract_dir.glob("*.nc"))
print("Extracted:", nc_file)

['reanalysis-era5-single-levels-timeseries-sfcfm6jb6mn.nc']
Extracted: /Users/sristykala/ppa_selection_italy/data/raw/weather/era5_test_torino_extracted/reanalysis-era5-single-levels-timeseries-sfcfm6jb6mn.nc


In [4]:
import xarray as xr

ds = xr.open_dataset(nc_file)
print(ds)
print()
print("Row count:", ds.sizes["valid_time"])
print("Expected (33 yrs, ~8 leap):", 33 * 365 * 24 + 8 * 24)
print()
print("Time range:", ds.valid_time.min().values, "to", ds.valid_time.max().values)


<xarray.Dataset> Size: 9MB
Dimensions:     (valid_time: 289272)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2MB 1991-01-01 ... 2023-12-31T23:...
    latitude    float64 8B ...
    longitude   float64 8B ...
Data variables:
    u100        (valid_time) float32 1MB ...
    v100        (valid_time) float32 1MB ...
    t2m         (valid_time) float32 1MB ...
    sp          (valid_time) float32 1MB ...
    ssrd        (valid_time) float32 1MB ...
    fdir        (valid_time) float32 1MB ...
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    history:                 2024-09-02T04:48 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

Row count: 289272
Expected (33 yrs, ~8 leap): 289272

Time range: 1991-01-01T00:00:00.000000000 to 2023-12-3